# 🧭 Linear Agent — LangGraph + Langfuse

> Notebook 1/2 cho lớp GenAI. Cặp với `02_orchestrator_agent.ipynb`.

## Linear là gì?

Các node chạy **lần lượt theo đường cố định**, đóng đinh lúc dựng graph:

```
START → trip_analyzer → destination_planner → itinerary_builder → END
```

**Ưu:** đơn giản, dễ đoán, dễ debug, rẻ (không có LLM call để định tuyến).
**Nhược:** cứng nhắc, không tự điều chỉnh.

👉 Mọi cell ở 2 notebook **giống hệt nhau**, chỉ khác phần **"7. Dựng graph"**.

## 1. Cài đặt & cấu hình

Project dùng [uv](https://docs.astral.sh/uv/). Mở terminal tại thư mục dự án:

```bash
uv sync                 # tạo .venv + cài đúng phiên bản thư viện
cp .env.example .env    # rồi điền OPENAI_API_KEY và (tuỳ chọn) LANGFUSE_* vào .env
```

Chọn kernel `.venv` cho notebook (góc trên phải VS Code → *Select Kernel*).

> Nếu KHÔNG dùng uv, bỏ comment dòng `%pip install` bên dưới.

In [ ]:
# Đã `uv sync` thì bỏ qua. Nếu chưa, bỏ comment để cài bằng pip:
# %pip install langchain==1.0.2 langchain-openai==1.0.1 langgraph==1.0.1 langfuse==3.11.2 python-dotenv

In [ ]:
import warnings
# Lọc vài cảnh báo vô hại cho gọn output (kết quả vẫn đúng)
warnings.filterwarnings("ignore", message="Pydantic serializer warnings")
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")

import os
import json
from typing import TypedDict, Optional, Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END

from langfuse import Langfuse, get_client
from langfuse.langchain import CallbackHandler

from IPython.display import Image, Markdown, display

load_dotenv()
print("Imports OK")

## 2. LLM + Langfuse (observability)

- `llm`: chat model (mặc định `gpt-4o-mini`).
- `langfuse_handler`: callback ghi MỌI lời gọi LLM/tool lên Langfuse. Ta truyền
  `config={"callbacks": [langfuse_handler]}` ở mỗi `invoke`.

Langfuse là **tuỳ chọn**: thiếu/sai key thì agent vẫn chạy, chỉ là không có trace.

In [ ]:
# --- Cấu hình ---
MODEL_NAME = "gpt-4o-mini"
TEMPERATURE = 0.7

# --- LLM ---
llm = init_chat_model(model=MODEL_NAME, temperature=TEMPERATURE)

# --- Langfuse: phải khởi tạo global client trước, rồi tạo CallbackHandler ---
_host = os.getenv("LANGFUSE_HOST") or os.getenv("LANGFUSE_BASE_URL") or "https://cloud.langfuse.com"
Langfuse(
    public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
    host=_host,
)
langfuse_handler = CallbackHandler()

try:
    if get_client().auth_check():
        print("✅ Langfuse đã kết nối:", _host)
except Exception:
    print("⚠️  Chưa kết nối Langfuse (kiểm tra key/host trong .env). Agent vẫn chạy, chỉ là không có trace.")

print("Model:", MODEL_NAME)

## 3. State — "bộ nhớ chung" của graph

Mọi node ĐỌC từ `State` chung và TRẢ VỀ phần cập nhật để LangGraph merge lại.
Cả hai kiến trúc dùng **đúng cùng một** `TravelState` — điểm khác biệt chỉ nằm ở *cách nối node*.

In [ ]:
class TravelState(TypedDict):
    """State object cho Travel Planning Agent (dùng chung Linear & Orchestrator)."""
    user_query: str                    # đầu vào của người dùng

    trip_analysis: Optional[dict]      # Node 1 -> kết quả
    destination_plan: Optional[dict]   # Node 2 -> kết quả
    itinerary: Optional[dict]          # Node 3 -> kết quả

    next_step: str                     # dùng cho Orchestrator: bước tiếp theo
    final_answer: str                  # câu trả lời tổng hợp cuối

    current_node: str                  # để debug/hiển thị
    errors: list[str]


def create_initial_travel_state(user_query: str) -> TravelState:
    return TravelState(
        user_query=user_query,
        trip_analysis=None, destination_plan=None, itinerary=None,
        next_step="", final_answer="", current_node="start", errors=[],
    )

print("TravelState gồm:", list(TravelState.__annotations__))

## 4. Node 1 — Trip Analyzer (ReAct agent + 5 tools)

Phân tích nhu cầu theo **2 bước cho chắc chắn**:
1. `create_agent` chạy vòng lặp ReAct, gọi 5 tool để thu thập thông tin (thời tiết, visa, ngân sách...).
2. `with_structured_output(TripAnalysis)` ép kết quả về **dữ liệu có cấu trúc** (Pydantic), không phải text tự do.

Trước hết, định nghĩa 5 tool (ở đây dùng dữ liệu giả lập để chạy offline):

In [ ]:
# ===== TOOL DEFINITIONS =====


@tool
def parse_travel_details(query: str) -> dict:
    """
    Extract travel details from user's natural language query.

    Args:
        query: User's travel planning request in natural language

    Returns:
        Dictionary with extracted details: destination, duration, travelers, timeframe, interests, budget
    """
    # Simulated extraction (in production, would use NLP/LLM parsing)
    # For now, return mock data based on query keywords

    result = {
        "destination": "Unknown",
        "duration": "Not specified",
        "travelers": "Not specified",
        "timeframe": "Not specified",
        "interests": [],
        "budget": "Not specified",
    }

    query_lower = query.lower()

    # Simple keyword-based extraction (demo purposes)
    if "tokyo" in query_lower or "japan" in query_lower:
        result["destination"] = "Tokyo, Japan"
    elif "paris" in query_lower or "france" in query_lower:
        result["destination"] = "Paris, France"
    elif "bali" in query_lower or "indonesia" in query_lower:
        result["destination"] = "Bali, Indonesia"
    elif "new york" in query_lower or "nyc" in query_lower:
        result["destination"] = "New York, USA"

    if "5 days" in query_lower or "five days" in query_lower:
        result["duration"] = "5 days"
    elif "week" in query_lower or "7 days" in query_lower:
        result["duration"] = "7 days"
    elif "3 days" in query_lower or "weekend" in query_lower:
        result["duration"] = "3 days"

    if "wife" in query_lower or "couple" in query_lower:
        result["travelers"] = "2 adults"
    elif "family" in query_lower:
        result["travelers"] = "Family (2 adults, 2 children assumed)"
    elif "solo" in query_lower or "alone" in query_lower:
        result["travelers"] = "1 adult"

    if "march" in query_lower:
        result["timeframe"] = "March 2026"
    elif "summer" in query_lower:
        result["timeframe"] = "Summer 2026"
    elif "december" in query_lower or "winter" in query_lower:
        result["timeframe"] = "December 2026"

    if "food" in query_lower or "cuisine" in query_lower:
        result["interests"].append("food")
    if "culture" in query_lower or "history" in query_lower:
        result["interests"].append("culture")
    if "beach" in query_lower or "relax" in query_lower:
        result["interests"].append("beach/relaxation")
    if "adventure" in query_lower or "hiking" in query_lower:
        result["interests"].append("adventure")

    return result


@tool
def check_weather(destination: str, month: str) -> dict:
    """
    Get weather forecast for destination and month.

    Args:
        destination: City/country name
        month: Month and year (e.g., "March 2026")

    Returns:
        Dictionary with temperature, conditions, and special notes
    """
    # Simulated weather data
    weather_db = {
        "Tokyo, Japan": {
            "March": {
                "temperature": "8-15°C (46-59°F)",
                "conditions": "Cool, occasional rain",
                "special": "Late March is cherry blossom season - high crowds and prices",
            },
            "Summer": {
                "temperature": "25-32°C (77-90°F)",
                "conditions": "Hot and humid, rainy season",
                "special": "Typhoon season, very humid",
            },
        },
        "Paris, France": {
            "March": {
                "temperature": "5-12°C (41-54°F)",
                "conditions": "Cool, rainy",
                "special": "Spring is starting, fewer tourists than summer",
            },
            "Summer": {
                "temperature": "18-25°C (64-77°F)",
                "conditions": "Pleasant, occasional rain",
                "special": "Peak tourist season, book ahead",
            },
        },
        "Bali, Indonesia": {
            "March": {
                "temperature": "26-31°C (79-88°F)",
                "conditions": "Warm, end of rainy season",
                "special": "Transitioning to dry season, good time to visit",
            },
            "December": {
                "temperature": "25-30°C (77-86°F)",
                "conditions": "Rainy season",
                "special": "Afternoon rains common, lush and green",
            },
        },
    }

    # Simple lookup
    for dest_key in weather_db:
        if destination in dest_key or dest_key in destination:
            for month_key in weather_db[dest_key]:
                if month_key.lower() in month.lower():
                    return weather_db[dest_key][month_key]

    # Default if not found
    return {
        "temperature": "Not available",
        "conditions": "Please check local weather services",
        "special": "No special notes",
    }


@tool
def check_travel_requirements(destination: str, travelers: str) -> dict:
    """
    Check visa requirements, vaccinations, and travel advisories.

    Args:
        destination: Country/city
        travelers: Description of travelers (e.g., "2 US citizens")

    Returns:
        Dictionary with visa, vaccination, and travel advisory information
    """
    # Simulated requirements database
    requirements_db = {
        "Japan": {
            "visa": "Not required for US/EU citizens for stays under 90 days",
            "vaccinations": "None required, routine vaccines recommended",
            "travel_advisory": "Level 1 - Exercise normal precautions",
        },
        "France": {
            "visa": "Not required for US citizens for stays under 90 days",
            "vaccinations": "None required, routine vaccines recommended",
            "travel_advisory": "Level 2 - Exercise increased caution",
        },
        "Indonesia": {
            "visa": "Visa on arrival for most nationalities, 30 days",
            "vaccinations": "Hepatitis A and Typhoid recommended",
            "travel_advisory": "Level 2 - Exercise increased caution",
        },
    }

    # Simple lookup
    for country in requirements_db:
        if country.lower() in destination.lower():
            return requirements_db[country]

    return {
        "visa": "Please check with embassy",
        "vaccinations": "Consult travel clinic",
        "travel_advisory": "Check government travel website",
    }


@tool
def estimate_budget(
    destination: str, days: int, travelers: int, style: str = "mid-range"
) -> dict:
    """
    Estimate trip budget based on destination, duration, and travel style.

    Args:
        destination: City/country
        days: Number of days
        travelers: Number of travelers
        style: Travel style (budget/mid-range/luxury/food_focused)

    Returns:
        Dictionary with cost breakdown and total estimate
    """
    # Base daily costs per destination (per person, mid-range)
    base_costs = {
        "Tokyo": {
            "daily_food": 60,
            "accommodation": 120,
            "activities": 40,
            "transport": 20,
        },
        "Paris": {
            "daily_food": 70,
            "accommodation": 140,
            "activities": 50,
            "transport": 15,
        },
        "Bali": {
            "daily_food": 30,
            "accommodation": 60,
            "activities": 35,
            "transport": 10,
        },
        "New York": {
            "daily_food": 80,
            "accommodation": 180,
            "activities": 60,
            "transport": 30,
        },
    }

    # Find matching destination
    dest_key = "Tokyo"  # default
    for key in base_costs:
        if key.lower() in destination.lower():
            dest_key = key
            break

    costs = base_costs[dest_key]

    # Adjust for style
    multipliers = {"budget": 0.6, "mid-range": 1.0, "luxury": 2.0, "food_focused": 1.3}
    multiplier = multipliers.get(style.replace("-", "_"), 1.0)

    # Calculate per person
    accommodation_total = costs["accommodation"] * days * multiplier
    food_total = costs["daily_food"] * days * multiplier
    if "food" in style:
        food_total *= 1.5  # Extra budget for food experiences

    activities_total = costs["activities"] * days * multiplier
    transport_total = costs["transport"] * days

    # Rough flight estimate
    flight_estimates = {"Tokyo": 1200, "Paris": 800, "Bali": 1000, "New York": 400}
    flights = flight_estimates.get(dest_key, 800)

    total_per_person = (
        flights + accommodation_total + food_total + activities_total + transport_total
    )

    return {
        "flights": f"${flights:,.0f}",
        "accommodation": f"${accommodation_total:,.0f}",
        "food": f"${food_total:,.0f}",
        "activities": f"${activities_total:,.0f}",
        "transport": f"${transport_total:,.0f}",
        "total_estimate": f"${total_per_person:,.0f} per person",
        "total_for_group": f"${total_per_person * travelers:,.0f} total",
    }


@tool
def validate_trip_feasibility(destination: str, month: str, duration: str) -> dict:
    """
    Validate if the trip timing and duration are feasible and optimal.

    Args:
        destination: City/country
        month: Month of travel
        duration: Duration of trip

    Returns:
        Dictionary with feasibility assessment, pros, cons, and recommendations
    """
    # Extract number of days
    days = 5  # default
    if "3" in duration:
        days = 3
    elif "5" in duration:
        days = 5
    elif "7" in duration or "week" in duration:
        days = 7
    elif "10" in duration:
        days = 10

    # Assess based on destination
    assessments = {
        "Tokyo": {
            "ideal_days": 5,
            "min_days": 3,
            "max_days": 14,
            "best_months": ["March", "April", "October", "November"],
            "avoid_months": ["August"],
        },
        "Paris": {
            "ideal_days": 5,
            "min_days": 3,
            "max_days": 10,
            "best_months": ["April", "May", "September", "October"],
            "avoid_months": ["August"],
        },
        "Bali": {
            "ideal_days": 7,
            "min_days": 5,
            "max_days": 14,
            "best_months": ["April", "May", "June", "September"],
            "avoid_months": ["January", "February"],
        },
    }

    # Find matching destination
    dest_key = "Tokyo"
    for key in assessments:
        if key.lower() in destination.lower():
            dest_key = key
            break

    info = assessments[dest_key]

    # Assess feasibility
    feasibility = "GOOD"
    pros = []
    cons = []

    if days >= info["min_days"] and days <= info["ideal_days"] + 2:
        pros.append(f"{days} days is a great duration for {dest_key}")
    elif days < info["min_days"]:
        cons.append(f"{days} days is quite short, consider {info['ideal_days']} days")
        feasibility = "TIGHT"
    elif days > info["max_days"]:
        pros.append(f"{days} days allows for deep exploration")

    # Check month
    month_name = month.split()[0] if month else ""
    if any(m in month_name for m in info["best_months"]):
        pros.append(f"{month_name} is an excellent time to visit")
    if any(m in month_name for m in info["avoid_months"]):
        cons.append(f"{month_name} can be challenging (weather/crowds)")
        feasibility = "ACCEPTABLE" if feasibility == "GOOD" else feasibility

    if not cons:
        cons.append("No major concerns")

    return {
        "feasibility": feasibility,
        "pros": pros,
        "cons": cons,
        "recommendation": f"Trip is {feasibility.lower()} - {info['ideal_days']} days is ideal for {dest_key}",
    }


In [ ]:
# ===== STRUCTURED OUTPUT SCHEMA =====
# Quy trình 2 BƯỚC cho chắc chắn:
#   (1) ReAct agent gọi tool để THU THẬP thông tin (tự kết thúc khi đủ),
#   (2) with_structured_output ÉP kết quả về đúng schema -> dữ liệu thật, không phải text.


class TripSummary(BaseModel):
    destination: str = Field(description="Điểm đến, ví dụ 'Tokyo, Japan'")
    dates: str = Field(description="Thời điểm đi, ví dụ 'March 2026'")
    travelers: str = Field(description="Số người đi, ví dụ '2 adults'")
    duration: str = Field(description="Thời lượng, ví dụ '5 days'")


class TripAnalysis(BaseModel):
    """Kết quả phân tích chuyến đi do Trip Analyzer tạo ra."""
    trip_summary: TripSummary
    feasibility: str = Field(description="Mức độ khả thi: GOOD / ACCEPTABLE / TIGHT")
    weather: str = Field(description="Tóm tắt thời tiết tại điểm đến trong thời gian đi")
    requirements: str = Field(description="Visa, vaccine, lưu ý nhập cảnh")
    budget_estimate: str = Field(description="Khoảng ngân sách ước tính")
    key_insights: list[str] = Field(description="Các nhận định quan trọng về chuyến đi")
    questions_for_user: list[str] = Field(description="Câu hỏi cần làm rõ (nếu có)")


# ===== NODE FUNCTION =====

SYSTEM_PROMPT = """Bạn là Trip Analyzer - chuyên gia phân tích nhu cầu du lịch.

Hãy DÙNG các tool để thu thập thông tin: trích xuất chi tiết chuyến đi, kiểm tra thời tiết,
visa/yêu cầu nhập cảnh, ước tính ngân sách, đánh giá tính khả thi. Khi đã đủ thông tin,
tóm tắt lại thành bản phân tích rõ ràng."""


def trip_analyzer_node(state: TravelState) -> dict:
    """Node 1 - ReAct agent (5 tools) nghiên cứu, rồi trích xuất structured output."""
    user_query = state["user_query"]
    tools = [parse_travel_details, check_weather, check_travel_requirements,
             estimate_budget, validate_trip_feasibility]
    try:
        # Bước 1: ReAct agent gọi tool để nghiên cứu (tự kết thúc khi đủ)
        agent = create_agent(model=llm, tools=tools, system_prompt=SYSTEM_PROMPT)
        res = agent.invoke(
            {"messages": [HumanMessage(content=f"Phân tích yêu cầu du lịch: {user_query}")]},
            config={"callbacks": [langfuse_handler], "recursion_limit": 50},
        )
        research = res["messages"][-1].content

        # Bước 2: ép phần nghiên cứu về đúng schema
        analysis = llm.with_structured_output(TripAnalysis).invoke(
            [SystemMessage(content="Trích xuất bản phân tích theo schema, chỉ dùng thông tin có trong phần nghiên cứu dưới đây."),
             HumanMessage(content=research)],
            config={"callbacks": [langfuse_handler]},
        )
        return {"trip_analysis": analysis.model_dump(), "current_node": "destination_planner"}
    except Exception as e:
        return {
            "trip_analysis": {"error": str(e), "trip_summary": {}, "feasibility": "UNKNOWN",
                              "weather": "N/A", "requirements": "N/A", "budget_estimate": "N/A",
                              "key_insights": [], "questions_for_user": []},
            "current_node": "destination_planner",
            "errors": state.get("errors", []) + [f"Trip analyzer error: {e}"],
        }


## 5. Node 2 — Destination Planner (ReAct agent + 5 tools)

Dựa trên `trip_analysis`, tìm chỗ ở / ẩm thực / điểm tham quan — cũng theo **2 bước**:
ReAct agent gọi tool nghiên cứu, rồi `with_structured_output(DestinationPlan)` ép kết quả về schema.

5 tool của node này:

In [ ]:
# ===== TOOL DEFINITIONS =====


@tool
def search_attractions(
    city: str, categories: list[str], time_per_visit: str = "2-3 hours"
) -> dict:
    """
    Find tourist attractions by category.

    Args:
        city: City name
        categories: List of categories (e.g., culture, iconic, nature, instagram_worthy)
        time_per_visit: Expected time per visit

    Returns:
        Dictionary with list of top attractions matching criteria
    """
    # Simulated attractions database
    attractions_db = {
        "Tokyo": [
            {
                "name": "Senso-ji Temple",
                "categories": ["culture", "iconic", "historic"],
                "description": "Tokyo's oldest temple, beautiful architecture",
                "time": "2 hours",
                "cost": "Free",
                "rating": 4.5,
            },
            {
                "name": "Meiji Shrine",
                "categories": ["culture", "nature", "peaceful"],
                "description": "Peaceful shrine in forest setting",
                "time": "1.5 hours",
                "cost": "Free",
                "rating": 4.6,
            },
            {
                "name": "teamLab Borderless",
                "categories": ["art", "instagram_worthy", "modern"],
                "description": "Mind-blowing digital art museum",
                "time": "2-3 hours",
                "cost": "$30",
                "rating": 4.8,
            },
            {
                "name": "Tokyo Skytree",
                "categories": ["iconic", "views", "instagram_worthy"],
                "description": "Tallest tower in Japan, amazing views",
                "time": "2 hours",
                "cost": "$25",
                "rating": 4.4,
            },
            {
                "name": "Harajuku/Takeshita Street",
                "categories": ["culture", "shopping", "instagram_worthy"],
                "description": "Youth culture, fashion, street food",
                "time": "3 hours",
                "cost": "Free (+ shopping)",
                "rating": 4.3,
            },
        ],
        "Paris": [
            {
                "name": "Eiffel Tower",
                "categories": ["iconic", "views", "instagram_worthy"],
                "description": "Iconic iron tower, symbol of Paris",
                "time": "2-3 hours",
                "cost": "$20-35",
                "rating": 4.7,
            },
            {
                "name": "Louvre Museum",
                "categories": ["culture", "art", "iconic"],
                "description": "World's largest art museum",
                "time": "3-4 hours",
                "cost": "$18",
                "rating": 4.8,
            },
        ],
    }

    # Get attractions for city
    city_attractions = attractions_db.get(city, [])

    # Filter by categories
    matching = []
    for attr in city_attractions:
        if any(cat in attr["categories"] for cat in categories):
            matching.append(attr)

    return {
        "city": city,
        "found_count": len(matching),
        "top_attractions": matching[:5],  # Return top 5
        "recommendation": f"Found {len(matching)} attractions matching your interests",
    }


@tool
def search_restaurants(
    city: str, interests: list[str], budget: str = "mid-range"
) -> dict:
    """
    Find restaurants by cuisine and rating.

    Args:
        city: City name
        interests: List of interests (e.g., authentic, variety, local_favorites, michelin)
        budget: Budget level (budget/mid-range/high-end)

    Returns:
        Dictionary with list of top restaurant experiences
    """
    # Simulated restaurant database
    restaurants_db = {
        "Tokyo": [
            {
                "name": "Tsukiji Outer Market",
                "type": "Fresh seafood breakfast, street food",
                "cuisine": "Japanese seafood",
                "experience": "authentic",
                "price_range": "$20-40",
                "rating": 4.7,
                "tip": "Go early (6-7am) for best selection",
            },
            {
                "name": "Sushi Dai",
                "type": "Premium sushi counter",
                "cuisine": "Sushi",
                "experience": "authentic",
                "price_range": "$50-60",
                "rating": 4.8,
                "tip": "Arrive by 5am or expect 3+ hour wait - totally worth it!",
            },
            {
                "name": "Golden Gai Izakayas",
                "type": "Tiny bars, local atmosphere",
                "cuisine": "Izakaya (Japanese pub food)",
                "experience": "local_favorites",
                "price_range": "$40-60",
                "rating": 4.5,
                "tip": "Bar-hop through several, each fits only 5-7 people",
            },
            {
                "name": "Ramen Street (Tokyo Station)",
                "type": "Famous ramen shops",
                "cuisine": "Ramen",
                "experience": "variety",
                "price_range": "$12-18",
                "rating": 4.6,
                "tip": "8 different ramen shops to choose from",
            },
            {
                "name": "Yakitori Alley (Yurakucho)",
                "type": "Grilled chicken skewers under train tracks",
                "cuisine": "Yakitori",
                "experience": "local_favorites",
                "price_range": "$30-50",
                "rating": 4.4,
                "tip": "Atmospheric spot under elevated tracks",
            },
        ],
        "Paris": [
            {
                "name": "L'Avant Comptoir",
                "type": "Standing wine bar, small plates",
                "cuisine": "French tapas",
                "experience": "local_favorites",
                "price_range": "$40-60",
                "rating": 4.6,
                "tip": "No reservations, arrive early",
            },
        ],
    }

    # Get restaurants for city
    city_restaurants = restaurants_db.get(city, [])

    # Filter by interests
    matching = []
    for rest in city_restaurants:
        if any(interest in rest["experience"] for interest in interests):
            matching.append(rest)

    # If no matches, return all
    if not matching:
        matching = city_restaurants

    return {
        "city": city,
        "found_count": len(matching),
        "top_food_experiences": matching[:5],
        "recommendation": f"Found {len(matching)} great food experiences",
    }


@tool
def search_hotels(
    city: str, budget: str = "mid-range", preferences: list[str] = None
) -> dict:
    """
    Find accommodation options.

    Args:
        city: City name
        budget: Budget level (budget/mid-range/luxury)
        preferences: List of preferences (e.g., central_location, near_food_areas, quiet)

    Returns:
        Dictionary with hotel recommendations
    """
    if preferences is None:
        preferences = []

    # Simulated hotel database
    hotels_db = {
        "Tokyo": [
            {
                "name": "Shinjuku Granbell Hotel",
                "location": "Shinjuku",
                "price_per_night": 150,
                "budget_level": "mid-range",
                "features": ["central_location", "near_food_areas", "metro_access"],
                "rating": 4.5,
                "description": "Perfect location near train lines, walking distance to restaurants",
                "booking_url": "https://booking.example.com/shinjuku-granbell",
            },
            {
                "name": "Hotel Gracery Shinjuku",
                "location": "Shinjuku",
                "price_per_night": 130,
                "budget_level": "mid-range",
                "features": ["central_location", "metro_access"],
                "rating": 4.4,
                "description": "Famous for Godzilla head on rooftop",
                "booking_url": "https://booking.example.com/gracery",
            },
            {
                "name": "Shibuya Excel Hotel Tokyu",
                "location": "Shibuya",
                "price_per_night": 160,
                "budget_level": "mid-range",
                "features": ["central_location", "trendy", "metro_access"],
                "rating": 4.6,
                "description": "Right above Shibuya station, unbeatable convenience",
                "booking_url": "https://booking.example.com/shibuya-excel",
            },
        ],
        "Paris": [
            {
                "name": "Hotel Les Dames du Panthéon",
                "location": "Latin Quarter",
                "price_per_night": 180,
                "budget_level": "mid-range",
                "features": ["central_location", "charming", "quiet"],
                "rating": 4.7,
                "description": "Charming boutique hotel in Latin Quarter",
                "booking_url": "https://booking.example.com/les-dames",
            },
        ],
    }

    # Get hotels for city
    city_hotels = hotels_db.get(city, [])

    # Filter by budget and preferences
    matching = []
    for hotel in city_hotels:
        if hotel["budget_level"] == budget or not budget:
            if not preferences or any(
                pref in hotel["features"] for pref in preferences
            ):
                matching.append(hotel)

    # Sort by rating
    matching.sort(key=lambda x: x["rating"], reverse=True)

    return {
        "city": city,
        "found_count": len(matching),
        "top_pick": matching[0] if matching else None,
        "other_options": matching[1:4] if len(matching) > 1 else [],
        "recommendation": (
            f"Top recommendation: {matching[0]['name']}"
            if matching
            else "No hotels found"
        ),
    }


@tool
def get_location_details(place: str) -> dict:
    """
    Get detailed information about a specific place.

    Args:
        place: Name of place (restaurant, attraction, etc.)

    Returns:
        Detailed information including cost, wait times, experience description
    """
    # Simulated detailed info database
    details_db = {
        "Sushi Dai, Tsukiji": {
            "name": "Sushi Dai",
            "type": "Authentic sushi counter",
            "cost": "$40-60 per person",
            "wait_time": "2-4 hours (arrive before 5am)",
            "experience": "Omakase-style, ultra fresh from market",
            "rating": 4.8,
            "worth_it": "Yes - iconic Tokyo experience",
            "insider_tip": "The wait is part of the experience. Bring a book or chat with other tourists in line.",
        },
        "teamLab Borderless": {
            "name": "teamLab Borderless",
            "type": "Digital art museum",
            "cost": "$30 per person",
            "wait_time": "Book tickets online to avoid lines",
            "experience": "Immersive digital art installations, Instagram heaven",
            "rating": 4.8,
            "worth_it": "Absolutely - unique experience",
            "insider_tip": "Go on weekday mornings for fewer crowds. Wear comfortable shoes.",
        },
    }

    # Simple lookup
    for key in details_db:
        if place.lower() in key.lower() or key.lower() in place.lower():
            return details_db[key]

    return {
        "name": place,
        "details": "Details not available in database",
        "recommendation": "Check online reviews for current information",
    }


@tool
def calculate_distances(start: str, destinations: list[str]) -> dict:
    """
    Calculate travel time between locations.

    Args:
        start: Starting location (e.g., hotel name)
        destinations: List of destination names

    Returns:
        Dictionary with travel times to each destination
    """
    # Simulated distance/time database
    # In reality, would use Google Maps API or similar

    # Shinjuku as reference point in Tokyo
    shinjuku_times = {
        "Tsukiji Market": {"time": "30 min", "method": "metro"},
        "Senso-ji": {"time": "35 min", "method": "metro"},
        "Shibuya": {"time": "10 min", "method": "metro"},
        "Tokyo Station": {"time": "15 min", "method": "metro"},
        "Harajuku": {"time": "8 min", "method": "metro"},
        "teamLab Borderless": {"time": "40 min", "method": "metro"},
        "Meiji Shrine": {"time": "12 min", "method": "metro"},
    }

    results = {}
    for dest in destinations:
        # Simple lookup
        found = False
        for key in shinjuku_times:
            if key.lower() in dest.lower() or dest.lower() in key.lower():
                results[dest] = shinjuku_times[key]
                found = True
                break
        if not found:
            results[dest] = {"time": "Unknown", "method": "Check Google Maps"}

    assessment = "Excellent location - all major spots easily accessible"
    if all(
        "min" in r.get("time", "") and int(r["time"].split()[0]) <= 35
        for r in results.values()
        if "min" in r.get("time", "")
    ):
        assessment = "Excellent location - all major spots within 35min"

    return {"start": start, "travel_times": results, "assessment": assessment}


In [ ]:
# ===== STRUCTURED OUTPUT SCHEMA =====


class Accommodation(BaseModel):
    name: str = Field(description="Tên khách sạn/chỗ ở")
    price: str = Field(description="Giá mỗi đêm, ví dụ '$150/night'")
    why: str = Field(description="Vì sao là lựa chọn tốt")


class FoodExperience(BaseModel):
    name: str = Field(description="Tên quán/trải nghiệm ẩm thực")
    experience: str = Field(description="Mô tả ngắn")
    cost: str = Field(description="Chi phí ước tính")
    tip: str = Field(description="Mẹo nhỏ")


class Attraction(BaseModel):
    name: str = Field(description="Tên điểm tham quan")
    why: str = Field(description="Lý do nên đến")
    time: str = Field(description="Thời gian nên dành")
    cost: str = Field(description="Chi phí vào cửa")


class DestinationPlan(BaseModel):
    """Kế hoạch điểm đến do Destination Planner tạo ra."""
    accommodation: Accommodation
    must_do_food: list[FoodExperience] = Field(description="3-5 trải nghiệm ẩm thực")
    top_attractions: list[Attraction] = Field(description="3-5 điểm tham quan")
    daily_budget: str = Field(description="Gợi ý ngân sách mỗi ngày cho mỗi người")


# ===== NODE FUNCTION =====


def destination_planner_node(state: TravelState) -> dict:
    """Node 2 - ReAct agent (5 tools) nghiên cứu, rồi trích xuất structured output."""
    trip_analysis = state.get("trip_analysis") or {}
    summary = trip_analysis.get("trip_summary", {})
    destination = summary.get("destination", "Tokyo, Japan")
    duration = summary.get("duration", "5 days")

    tools = [search_attractions, search_restaurants, search_hotels,
             get_location_details, calculate_distances]
    system_prompt = (
        f"Bạn là Destination Planner. Dùng các tool để tìm chỗ ở, ẩm thực và điểm tham quan "
        f"cho {destination} ({duration}). Khi đã đủ thông tin thì tổng hợp thành kế hoạch rõ ràng."
    )
    try:
        # Bước 1: ReAct agent nghiên cứu bằng tool (tự kết thúc)
        agent = create_agent(model=llm, tools=tools, system_prompt=system_prompt)
        res = agent.invoke(
            {"messages": [HumanMessage(content=f"Lập kế hoạch điểm đến cho {destination} trong {duration}.")]},
            config={"callbacks": [langfuse_handler], "recursion_limit": 50},
        )
        research = res["messages"][-1].content

        # Bước 2: ép về schema
        plan = llm.with_structured_output(DestinationPlan).invoke(
            [SystemMessage(content="Trích xuất kế hoạch điểm đến theo schema, chỉ dùng thông tin trong phần nghiên cứu dưới đây."),
             HumanMessage(content=research)],
            config={"callbacks": [langfuse_handler]},
        )
        return {"destination_plan": plan.model_dump(), "current_node": "itinerary_builder"}
    except Exception as e:
        return {
            "destination_plan": {"error": str(e), "accommodation": {}, "must_do_food": [],
                                 "top_attractions": [], "daily_budget": "N/A"},
            "current_node": "itinerary_builder",
            "errors": state.get("errors", []) + [f"Destination planner error: {e}"],
        }


## 6. Node 3 — Itinerary Builder (standard node)

Khác 2 node trên: đây là node "thường" — chỉ gọi LLM **một lần** (không tool) với structured output,
để tổng hợp thành lịch trình từng ngày.

In [ ]:
# ===== STRUCTURED OUTPUT SCHEMA =====


class DayPlan(BaseModel):
    day: int = Field(description="Ngày thứ mấy, ví dụ 1")
    theme: str = Field(description="Chủ đề của ngày, ví dụ 'Ẩm thực & văn hoá truyền thống'")
    morning: str = Field(description="Hoạt động buổi sáng (kèm giờ, chi phí nếu có)")
    afternoon: str = Field(description="Hoạt động buổi chiều")
    evening: str = Field(description="Hoạt động buổi tối")


class Itinerary(BaseModel):
    """Lịch trình hoàn chỉnh theo từng ngày."""

    days: list[DayPlan] = Field(description="Danh sách kế hoạch từng ngày")
    budget_summary: str = Field(description="Tóm tắt ngân sách (chỗ ở, ăn uống, hoạt động...)")
    pro_tips: list[str] = Field(description="Mẹo thực tế cho chuyến đi")


# ===== NODE FUNCTION =====


def _parse_num_days(duration: str) -> int:
    """Suy ra số ngày từ chuỗi duration (ví dụ '5 days' -> 5)."""
    duration = (duration or "").lower()
    for n in (10, 7, 5, 3):
        if str(n) in duration:
            return n
    if "week" in duration:
        return 7
    return 5


def itinerary_builder_node(state: TravelState) -> dict:
    """
    Node 3 - Itinerary Builder (standard node).

    Tổng hợp dữ liệu từ 2 bước trước thành lịch trình từng ngày (structured output).
    """
    trip_analysis = state.get("trip_analysis") or {}
    destination_plan = state.get("destination_plan") or {}

    trip_summary = trip_analysis.get("trip_summary", {})
    destination = trip_summary.get("destination", "Unknown")
    duration = trip_summary.get("duration", "5 days")
    num_days = _parse_num_days(duration)

    system_prompt = """Bạn là chuyên gia lập lịch trình du lịch. Hãy tạo lịch trình chi tiết theo từng ngày:

1. Phân bổ hoạt động hợp lý qua các ngày
2. Sắp xếp theo thời điểm tối ưu (chợ/ẩm thực buổi sáng, bảo tàng/tham quan buổi chiều, ăn tối buổi tối)
3. Cân bằng giữa ẩm thực và hoạt động văn hoá
4. Kèm chi phí, gợi ý di chuyển và mẹo nhỏ

Dùng đúng các gợi ý về chỗ ở/ẩm thực/điểm tham quan được cung cấp."""

    human_prompt = f"""Tạo lịch trình {num_days} ngày cho {destination}.

THÔNG TIN CHUYẾN ĐI:
{json.dumps(trip_summary, indent=2, ensure_ascii=False)}

KẾ HOẠCH ĐIỂM ĐẾN (chỗ ở, ẩm thực, tham quan):
{json.dumps(destination_plan, indent=2, ensure_ascii=False)}

Ngân sách ước tính: {trip_analysis.get("budget_estimate", "N/A")}

Hãy lồng ghép các gợi ý trên vào lịch trình một cách tự nhiên."""

    # Standard node: gọi LLM 1 lần, ép structured output theo schema Itinerary.
    structured_llm = llm.with_structured_output(Itinerary)

    try:
        itinerary: Itinerary = structured_llm.invoke(
            [SystemMessage(content=system_prompt), HumanMessage(content=human_prompt)],
            config={"callbacks": [langfuse_handler]},
        )

        return {
            "itinerary": itinerary.model_dump(),
            "current_node": "complete",
        }

    except Exception as e:
        return {
            "itinerary": {
                "error": f"Failed to build itinerary: {str(e)}",
                "days": [],
                "budget_summary": "N/A",
                "pro_tips": [],
            },
            "current_node": "complete",
            "errors": state.get("errors", []) + [f"Itinerary builder error: {str(e)}"],
        }


## 7. Dựng graph — LINEAR  ⭐ (phần DUY NHẤT khác notebook Orchestrator)

Nối node bằng `add_edge` cố định. Không có `add_conditional_edges`, không có "bộ não" định tuyến.

In [ ]:
graph = StateGraph(TravelState)

# 3 node công việc (giống hệt notebook Orchestrator)
graph.add_node("trip_analyzer", trip_analyzer_node)
graph.add_node("destination_planner", destination_planner_node)
graph.add_node("itinerary_builder", itinerary_builder_node)

##### TODO: Thực hành #####
# Yêu cầu:
#   - Nối các node thành một đường đi TUYẾN TÍNH bằng add_edge (cạnh CỐ ĐỊNH):
#       START -> trip_analyzer -> destination_planner -> itinerary_builder -> END
#   - Sau đó compile graph và gán vào biến `app`.
#   - Lưu ý: bản Linear KHÔNG dùng add_conditional_edges.
#######################
### START CODE HERE ###
#######################


##### End TODO #####

print("✅ Đã dựng LINEAR graph")

## 8. Trực quan hoá graph

`draw_mermaid_png()` cần mạng (gọi mermaid.ink). Nếu lỗi, in sơ đồ Mermaid dạng text.

In [ ]:
try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    print("Không vẽ được PNG (thường do mạng). Sơ đồ Mermaid:\n")
    print(app.get_graph().draw_mermaid())

## 9. Chạy thử

In [ ]:
query = "I want to visit Tokyo with my wife in March for 5 days, we love food"

final_state = app.invoke(
    create_initial_travel_state(query),
    config={"callbacks": [langfuse_handler]},
)
print("Các khoá trong state cuối:", list(final_state.keys()))

In [ ]:
# Orchestrator có câu trả lời tổng hợp; Linear có lịch trình chi tiết.
if final_state.get("final_answer"):
    display(Markdown(final_state["final_answer"]))
else:
    print(json.dumps(final_state["itinerary"], indent=2, ensure_ascii=False))

## 10. Xem trace trên Langfuse

`flush()` đẩy trace lên Langfuse. Mở dashboard để xem cây gọi LLM/tool, token, độ trễ từng node —
và so sánh số bước giữa Linear và Orchestrator.

In [ ]:
get_client().flush()
host = os.getenv("LANGFUSE_HOST") or os.getenv("LANGFUSE_BASE_URL") or "https://cloud.langfuse.com"
print("Mở dashboard Langfuse:", host)

## 11. Tổng kết — Linear

- Đường đi **cố định**, biết trước 100%.
- Chỉ tốn LLM call cho công việc thật (3 node), **không** có call định tuyến.
- Hợp với quy trình rõ ràng, ít rẽ nhánh.

➡️ Mở `02_orchestrator_agent.ipynb`: chỉ phần **"7. Dựng graph"** thay đổi. Trên Langfuse,
so sánh số bước/độ trễ giữa hai bản.